In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open("files/names.txt").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [4]:
len(words)

32033

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 
itos = {i:s for s,i in stoi.items()}

In [6]:
block_size = 3 # context lenght: how many characters do we take to predict the next one     
X,Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(context)
        print(''.join(itos[i] for i in context), "--->",itos[ix])
        context = context[1:] + [ix] # crop and update
        
X = torch.tensor(X)
Y = torch.tensor(Y)


emma
[0, 0, 0]
... ---> e
[0, 0, 5]
..e ---> m
[0, 5, 13]
.em ---> m
[5, 13, 13]
emm ---> a
[13, 13, 1]
mma ---> .
olivia
[0, 0, 0]
... ---> o
[0, 0, 15]
..o ---> l
[0, 15, 12]
.ol ---> i
[15, 12, 9]
oli ---> v
[12, 9, 22]
liv ---> i
[9, 22, 9]
ivi ---> a
[22, 9, 1]
via ---> .
ava
[0, 0, 0]
... ---> a
[0, 0, 1]
..a ---> v
[0, 1, 22]
.av ---> a
[1, 22, 1]
ava ---> .
isabella
[0, 0, 0]
... ---> i
[0, 0, 9]
..i ---> s
[0, 9, 19]
.is ---> a
[9, 19, 1]
isa ---> b
[19, 1, 2]
sab ---> e
[1, 2, 5]
abe ---> l
[2, 5, 12]
bel ---> l
[5, 12, 12]
ell ---> a
[12, 12, 1]
lla ---> .
sophia
[0, 0, 0]
... ---> s
[0, 0, 19]
..s ---> o
[0, 19, 15]
.so ---> p
[19, 15, 16]
sop ---> h
[15, 16, 8]
oph ---> i
[16, 8, 9]
phi ---> a
[8, 9, 1]
hia ---> .


In [8]:
X.shape,X.dtype,Y.shape,Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [9]:
C = torch.randn((27,2))

In [10]:
C[5]

tensor([-0.1556, -0.2370])

In [11]:
F.one_hot(torch.tensor(5),27).float() @ C

tensor([-0.1556, -0.2370])

In [12]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [13]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [14]:
emb * W1 + b1

RuntimeError: The size of tensor a (2) must match the size of tensor b (100) at non-singleton dimension 2

In [15]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([32, 6])

In [16]:
len(torch.unbind(emb,1))

3

In [17]:
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [18]:
emb.view(32,6)[0] == torch.cat(torch.unbind(emb,1),1)[0]

tensor([True, True, True, True, True, True])

In [19]:
h = torch.tanh(emb.view(32,6) @ W1 + b1)
h,h.shape

(tensor([[ 0.9460, -0.9856,  0.9999,  ..., -0.6198, -0.2665,  0.9964],
         [ 0.7015, -0.7296,  0.9993,  ..., -0.1361,  0.5565,  0.9979],
         [ 0.9872, -0.9917,  0.9808,  ..., -0.4871,  0.1758,  0.0371],
         ...,
         [-0.1852,  0.9402, -0.9983,  ..., -0.9748,  0.9751, -0.9921],
         [ 0.9999,  0.9677, -0.1278,  ...,  0.9987,  0.9688, -0.9585],
         [-0.8048, -0.9951, -0.4224,  ..., -0.9913,  0.9210, -0.5125]]),
 torch.Size([32, 100]))

In [21]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)
logits = h @ W2 + b2
logits.shape

torch.Size([32, 27])

In [22]:
counts = logits.exp()

In [25]:
probs = counts / counts.sum(1,True)
probs.shape

torch.Size([32, 27])

In [26]:
probs[torch.arange(32),Y]

tensor([9.4765e-02, 3.6133e-08, 7.7770e-01, 1.1178e-07, 2.2067e-10, 3.0654e-11,
        2.2140e-15, 7.3714e-09, 8.2435e-01, 7.6508e-09, 3.5614e-08, 2.3089e-05,
        3.4514e-02, 6.5799e-09, 1.4613e-07, 2.6943e-12, 9.0385e-10, 4.4842e-04,
        1.4121e-07, 4.0453e-12, 6.0163e-06, 3.2818e-14, 4.3081e-09, 2.3172e-05,
        3.2070e-09, 3.8842e-10, 3.0533e-08, 9.1300e-10, 2.6427e-11, 5.2349e-08,
        1.8770e-09, 7.6010e-07])

In [30]:
loss = -probs[torch.arange(32),Y].log().mean()
loss

tensor(17.0047)

In [31]:
# ---------- More Respected ----------------

In [32]:
X.shape,Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [47]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g)
W1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [ ]:
sum(p.nelement() for p in parameters)

3481

In [49]:
emb = C[X] # 32,3,2
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32,100)
logits = h @ W2 + b2
counts = logits.exp()
probs = counts/counts.sum(1,True)
loss = -probs[torch.arange(32),Y].log().mean()
loss

tensor(17.7697)